# 🧠 Customer Segmentation & Retention Analysis
## Exploratory Data Analysis (EDA) Notebook

---

**Dataset**: Online Retail II (UCI Machine Learning Repository)  
**Objective**: Understand customer behaviour, identify patterns, and prepare features for ML pipeline  

### 📋 Notebook Structure
1. Environment Setup & Data Loading
2. Data Quality Assessment
3. Univariate Analysis
4. Revenue & Sales Trends
5. Customer Behaviour Analysis
6. RFM Deep-Dive
7. Geographic Analysis
8. Product Analysis
9. Correlation & Feature Insights
10. Key Findings Summary

---

## 1️⃣ Environment Setup & Data Loading

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Plot style ─────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0F1117',
    'axes.facecolor'   : '#1A1D2E',
    'axes.edgecolor'   : '#30363D',
    'axes.labelcolor'  : '#C9D1D9',
    'xtick.color'      : '#C9D1D9',
    'ytick.color'      : '#C9D1D9',
    'text.color'       : '#C9D1D9',
    'grid.color'       : '#21262D',
    'grid.alpha'       : 0.5,
    'figure.dpi'       : 120,
    'font.family'      : 'DejaVu Sans',
})

PALETTE = ['#E63946','#457B9D','#2A9D8F','#E9C46A','#F4A261','#A8DADC','#6D6875','#264653']
print('✅ Libraries loaded successfully')

In [ ]:
from src.preprocessing import run_preprocessing, data_summary

DATA_PATH = '../data/online_retail_II.csv'
df_raw    = pd.read_csv(DATA_PATH, encoding='latin-1')
df        = run_preprocessing(DATA_PATH)

print(f'📦 Raw shape    : {df_raw.shape}')
print(f'✅ Clean shape  : {df.shape}')
print(f'🗑  Rows removed : {len(df_raw) - len(df):,}')
df.head()

## 2️⃣ Data Quality Assessment

In [ ]:
print('=' * 55)
print('  DATA QUALITY REPORT')
print('=' * 55)

# Missing values in raw data
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
quality_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
quality_df = quality_df[quality_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('\n🔍 Missing Values (Raw Data):')
print(quality_df.to_string())

print(f'\n📋 Duplicate rows   : {df_raw.duplicated().sum():,}')
print(f'📋 Cancelled orders : {df_raw["Invoice"].astype(str).str.startswith("C").sum():,}')
print(f'📋 Negative qty rows: {(df_raw["Quantity"] < 0).sum():,}')
print(f'📋 Zero price rows  : {(df_raw["Price"] <= 0).sum():,}')

# Dtypes
print('\n📊 Data Types:')
print(df.dtypes.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Missing value bar chart
missing_plot = df_raw.isnull().sum().sort_values(ascending=True)
axes[0].barh(missing_plot.index, missing_plot.values, color=PALETTE[0], alpha=0.85)
axes[0].set_title('Missing Values per Column (Raw Data)', fontweight='bold')
axes[0].set_xlabel('Count')

# Before vs After cleaning
categories = ['Total Rows', 'Valid Customers', 'Valid Orders']
before = [len(df_raw), df_raw['Customer ID'].dropna().nunique(), df_raw['Invoice'].nunique()]
after  = [len(df), df['customer_id'].nunique(), df['invoice_no'].nunique()]

x = np.arange(len(categories))
w = 0.35
axes[1].bar(x - w/2, before, w, label='Before Cleaning', color=PALETTE[0], alpha=0.85)
axes[1].bar(x + w/2, after,  w, label='After Cleaning',  color=PALETTE[2], alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(categories, rotation=15)
axes[1].set_title('Before vs After Cleaning', fontweight='bold')
axes[1].legend()
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

## 3️⃣ Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Univariate Distributions (Clean Data)', fontsize=15, fontweight='bold', y=1.01)

cols_info = [
    ('quantity',     'Quantity per Line',    PALETTE[0],  (0, 200)),
    ('unit_price',   'Unit Price (£)',        PALETTE[1],  (0, 50)),
    ('total_amount', 'Total Amount (£)',      PALETTE[2],  (0, 500)),
    ('hour',         'Hour of Purchase',      PALETTE[3],  None),
    ('day_of_week',  'Day of Week (0=Mon)',   PALETTE[4],  None),
]

for ax, (col, title, color, xlim) in zip(axes.flat, cols_info):
    data = df[col]
    if xlim:
        data = data[data.between(*xlim)]
    ax.hist(data, bins=50, color=color, alpha=0.85, edgecolor='none')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.axvline(data.mean(),   color='white', linestyle='--', linewidth=1.2, label=f'Mean={data.mean():.1f}')
    ax.axvline(data.median(), color='yellow',linestyle=':',  linewidth=1.2, label=f'Median={data.median():.1f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Country count
ax = axes.flat[5]
top_countries = df['country'].value_counts().head(10)
ax.barh(top_countries.index, top_countries.values, color=PALETTE[5], alpha=0.85)
ax.set_title('Top 10 Countries by Transactions', fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

print('\n📊 Descriptive Statistics:')
df[['quantity','unit_price','total_amount']].describe().round(2)

## 4️⃣ Revenue & Sales Trends

In [ ]:
# Monthly revenue & orders
monthly = df.groupby('year_month').agg(
    revenue        = ('total_amount', 'sum'),
    orders         = ('invoice_no',   'nunique'),
    unique_customers = ('customer_id','nunique'),
).reset_index()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Monthly Revenue (£)', 'Monthly Orders', 'Unique Customers/Month', 'Avg Order Value'],
)

monthly['aov'] = monthly['revenue'] / monthly['orders']

traces = [
    go.Scatter(x=monthly['year_month'], y=monthly['revenue'],        name='Revenue',    line=dict(color='#58A6FF', width=2.5), fill='tozeroy', fillcolor='rgba(88,166,255,0.1)'),
    go.Scatter(x=monthly['year_month'], y=monthly['orders'],         name='Orders',     line=dict(color='#2A9D8F', width=2.5)),
    go.Scatter(x=monthly['year_month'], y=monthly['unique_customers'],name='Customers', line=dict(color='#E9C46A', width=2.5)),
    go.Scatter(x=monthly['year_month'], y=monthly['aov'],            name='AOV',        line=dict(color='#E63946', width=2.5)),
]

positions = [(1,1),(1,2),(2,1),(2,2)]
for trace, (r,c) in zip(traces, positions):
    fig.add_trace(trace, row=r, col=c)

fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(28,33,40,0.8)',
    font=dict(color='#C9D1D9'), showlegend=False, height=550,
    title='Revenue & Sales Trends Over Time',
)
fig.show()

In [ ]:
# Day × Hour revenue heatmap
heatmap_data = df.groupby(['day_of_week','hour'])['total_amount'].sum().unstack(fill_value=0)
day_labels = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    heatmap_data,
    cmap       = 'Blues',
    ax         = ax,
    linewidths = 0.3,
    linecolor  = '#0F1117',
    fmt        = ',.0f',
    cbar_kws   = dict(label='Revenue (£)'),
    yticklabels= day_labels,
)
ax.set_title('Revenue Heatmap: Day of Week × Hour of Day', fontweight='bold', fontsize=13)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()
print('💡 Insight: Most revenue comes from Tuesday–Thursday, 10:00–14:00')

## 5️⃣ Customer Behaviour Analysis

In [ ]:
# Per-customer aggregation
customer_stats = df.groupby('customer_id').agg(
    total_orders    = ('invoice_no',   'nunique'),
    total_revenue   = ('total_amount', 'sum'),
    total_items     = ('quantity',     'sum'),
    avg_basket      = ('total_amount', 'mean'),
    first_purchase  = ('invoice_date', 'min'),
    last_purchase   = ('invoice_date', 'max'),
).reset_index()

customer_stats['lifetime_days'] = (
    customer_stats['last_purchase'] - customer_stats['first_purchase']
).dt.days

print('📊 Customer Statistics Summary')
print('─' * 50)
summary_cols = ['total_orders','total_revenue','total_items','avg_basket','lifetime_days']
customer_stats[summary_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Customer-Level Distributions', fontsize=14, fontweight='bold')

plots = [
    ('total_orders',  'Orders per Customer',    PALETTE[0], None, 'log'),
    ('total_revenue', 'Revenue per Customer(£)',PALETTE[1], None, 'log'),
    ('avg_basket',    'Avg Basket Value (£)',   PALETTE[2], (0, 500), None),
    ('lifetime_days', 'Customer Lifetime (days)',PALETTE[3], None, None),
]

for ax, (col, title, color, xlim, yscale) in zip(axes.flat, plots):
    data = customer_stats[col].dropna()
    if xlim:
        data = data[data.between(*xlim)]
    ax.hist(data, bins=60, color=color, alpha=0.85, edgecolor='none')
    if yscale:
        ax.set_yscale(yscale)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.axvline(data.mean(),   color='white',  linestyle='--', linewidth=1.2, label=f'μ={data.mean():.1f}')
    ax.axvline(data.median(), color='yellow', linestyle=':',  linewidth=1.2, label=f'M={data.median():.1f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Pareto Analysis: top 20% customers = ?% revenue
cust_sorted = customer_stats.sort_values('total_revenue', ascending=False).reset_index(drop=True)
cust_sorted['cum_revenue']  = cust_sorted['total_revenue'].cumsum()
cust_sorted['cum_pct_rev']  = cust_sorted['cum_revenue']  / cust_sorted['total_revenue'].sum() * 100
cust_sorted['cum_pct_cust'] = (cust_sorted.index + 1) / len(cust_sorted) * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cust_sorted['cum_pct_cust'], cust_sorted['cum_pct_rev'],
        color=PALETTE[0], linewidth=2.5, label='Cumulative Revenue')
ax.axvline(20, color='yellow', linestyle='--', linewidth=1.5, label='Top 20% Customers')
ax.axhline(80, color='cyan',   linestyle='--', linewidth=1.5, label='80% Revenue')
ax.fill_between(cust_sorted['cum_pct_cust'], cust_sorted['cum_pct_rev'],
                alpha=0.1, color=PALETTE[0])

# Find actual % revenue from top 20%
top20_rev = cust_sorted[cust_sorted['cum_pct_cust'] <= 20]['cum_pct_rev'].max()
ax.annotate(f'Top 20% customers\ncontribute {top20_rev:.1f}% revenue',
            xy=(20, top20_rev), xytext=(35, top20_rev - 15),
            arrowprops=dict(arrowstyle='->', color='white'),
            color='white', fontsize=11, fontweight='bold')

ax.set_title('Pareto Analysis: Customer Revenue Concentration', fontweight='bold', fontsize=13)
ax.set_xlabel('Cumulative % of Customers')
ax.set_ylabel('Cumulative % of Revenue')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'💡 Pareto Insight: Top 20% customers generate {top20_rev:.1f}% of total revenue')

## 6️⃣ RFM Deep-Dive

In [ ]:
from src.rfm import build_rfm_table, segment_summary

rfm = build_rfm_table(df)
print(f'✅ RFM table built: {rfm.shape}')
rfm.head()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('RFM Feature Distributions', fontsize=14, fontweight='bold')

rfm_plots = [
    ('recency',       'Recency (Days)',       PALETTE[0]),
    ('frequency',     'Frequency (Orders)',   PALETTE[1]),
    ('monetary',      'Monetary (£)',         PALETTE[2]),
    ('log_recency',   'Log(Recency)',         PALETTE[3]),
    ('log_frequency', 'Log(Frequency)',       PALETTE[4]),
    ('log_monetary',  'Log(Monetary)',        PALETTE[5]),
]

for ax, (col, title, color) in zip(axes.flat, rfm_plots):
    data = rfm[col]
    if col == 'monetary':
        data = data[data < data.quantile(0.99)]  # remove top 1% outliers for vis
    ax.hist(data, bins=50, color=color, alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.axvline(data.mean(),   color='white',  linestyle='--', linewidth=1.2)
    ax.axvline(data.median(), color='yellow', linestyle=':',  linewidth=1.2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('💡 Log transform significantly reduces right skew in all three RFM dimensions')

In [ ]:
# Correlation heatmap
corr_cols = ['recency','frequency','monetary','r_score','f_score','m_score','rfm_score','churn']
corr = rfm[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    ax=ax, mask=mask, linewidths=0.5, linecolor='#0F1117',
    vmin=-1, vmax=1, center=0,
    cbar_kws=dict(label='Correlation'),
)
ax.set_title('RFM Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Segment analysis
seg_sum = segment_summary(rfm)
print('📊 Customer Segment Summary')
print('=' * 70)
print(seg_sum.to_string(index=False))

# Segment donut
fig = px.sunburst(
    rfm, path=['segment'],
    title='Customer Segment Distribution',
    color='rfm_score',
    color_continuous_scale='RdYlGn',
)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', font_color='#C9D1D9')
fig.show()

In [ ]:
# RFM 3D scatter
sample = rfm.sample(min(3000, len(rfm)), random_state=42)

fig = px.scatter_3d(
    sample,
    x='log_recency', y='log_frequency', z='log_monetary',
    color='segment', opacity=0.7, size_max=5,
    title='3D RFM Space – Customer Segments',
    labels=dict(log_recency='Log Recency', log_frequency='Log Frequency', log_monetary='Log Monetary'),
)
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    font_color='#C9D1D9',
    height=600,
    scene=dict(
        bgcolor='rgba(28,33,40,0.9)',
        xaxis=dict(gridcolor='#30363D', color='#C9D1D9'),
        yaxis=dict(gridcolor='#30363D', color='#C9D1D9'),
        zaxis=dict(gridcolor='#30363D', color='#C9D1D9'),
    )
)
fig.show()

## 7️⃣ Geographic Analysis

In [ ]:
country_stats = df.groupby('country').agg(
    revenue   = ('total_amount',  'sum'),
    customers = ('customer_id',   'nunique'),
    orders    = ('invoice_no',    'nunique'),
).reset_index().sort_values('revenue', ascending=False)

country_stats['revenue_pct'] = (country_stats['revenue'] / country_stats['revenue'].sum() * 100).round(2)
country_stats['avg_rev_per_cust'] = (country_stats['revenue'] / country_stats['customers']).round(2)

print('🌍 Top 15 Countries by Revenue')
print(country_stats.head(15).to_string(index=False))

# Choropleth
fig = px.choropleth(
    country_stats,
    locations='country', locationmode='country names',
    color='revenue', hover_data=['customers','orders','revenue_pct'],
    color_continuous_scale='Blues',
    title='Global Revenue Distribution',
)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', font_color='#C9D1D9', height=450)
fig.show()

## 8️⃣ Product Analysis

In [ ]:
product_stats = df.groupby(['stock_code','description']).agg(
    revenue     = ('total_amount', 'sum'),
    qty_sold    = ('quantity',     'sum'),
    orders      = ('invoice_no',   'nunique'),
    unit_price  = ('unit_price',   'mean'),
).reset_index().sort_values('revenue', ascending=False)

product_stats['description'] = product_stats['description'].str[:40]

print(f'📦 Total unique products : {len(product_stats):,}')
print(f'📦 Products with > £1000 revenue: {(product_stats["revenue"] > 1000).sum():,}')
print('\n🏆 Top 10 Products:')
print(product_stats[['description','revenue','qty_sold','orders']].head(10).to_string(index=False))

# Treemap
top20 = product_stats.head(20)
fig = px.treemap(
    top20, path=['description'], values='revenue',
    color='qty_sold', color_continuous_scale='Blues',
    title='Top 20 Products – Revenue Treemap',
)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', font_color='#C9D1D9')
fig.show()

In [ ]:
# Quick Cohort Retention
df2 = df.copy()
df2['cohort_month'] = df2.groupby('customer_id')['invoice_date'].transform('min').dt.to_period('M')
df2['order_month']  = df2['invoice_date'].dt.to_period('M')
df2['cohort_index'] = (df2['order_month'] - df2['cohort_month']).apply(lambda x: x.n)

cohort_pivot = (
    df2.groupby(['cohort_month','cohort_index'])['customer_id']
    .nunique()
    .unstack(fill_value=0)
)
cohort_pct = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0).iloc[:, :13]

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    cohort_pct * 100, annot=True, fmt='.0f',
    cmap='Blues', ax=ax, linewidths=0.3, linecolor='#0F1117',
    cbar_kws=dict(label='Retention %'),
    xticklabels=[f'M+{i}' for i in cohort_pct.columns],
    yticklabels=[str(c) for c in cohort_pct.index],
)
ax.set_title('Monthly Cohort Retention Analysis (%)', fontweight='bold', fontsize=13)
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (First Purchase Month)')
plt.tight_layout()
plt.show()
print('💡 Insight: Average 30-day retention is around 20–30%. Key drop-off after month 1.')

## 9️⃣ Correlation & Feature Insights

In [ ]:
# Boxplots: RFM by churn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RFM Distribution: Churned vs Active Customers', fontsize=13, fontweight='bold')

for ax, col, title, color_map in zip(
    axes,
    ['log_recency', 'log_frequency', 'log_monetary'],
    ['Log Recency', 'Log Frequency', 'Log Monetary'],
    [{'Active': '#2A9D8F', 'Churned': '#E63946'}]*3
):
    rfm['churn_label'] = rfm['churn'].map({0: 'Active', 1: 'Churned'})
    rfm.boxplot(column=col, by='churn_label', ax=ax,
                patch_artist=True,
                boxprops=dict(facecolor=PALETTE[2], color=PALETTE[2]),
                whiskerprops=dict(color='white'),
                capprops=dict(color='white'),
                medianprops=dict(color='yellow', linewidth=2))
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)

plt.suptitle('RFM by Churn Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix (Plotly)
sample_rfm = rfm.sample(min(2000, len(rfm)), random_state=42)

fig = px.scatter_matrix(
    sample_rfm,
    dimensions=['log_recency','log_frequency','log_monetary'],
    color='churn_label',
    color_discrete_map={'Active': '#2A9D8F', 'Churned': '#E63946'},
    opacity=0.5,
    title='RFM Pair Plot – Churned vs Active',
    labels=dict(log_recency='Log R', log_frequency='Log F', log_monetary='Log M'),
)
fig.update_traces(diagonal_visible=True)
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    font_color='#C9D1D9',
    height=550,
)
fig.show()

## 🔟 Key Findings & Recommendations

In [ ]:
# Final summary stats
print('=' * 65)
print('  📋 KEY EDA FINDINGS SUMMARY')
print('=' * 65)

churn_rate = rfm['churn'].mean()
top_seg    = rfm['segment'].value_counts().idxmax()
top_country_rev = df.groupby('country')['total_amount'].sum().idxmax()
best_day   = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'][
    df.groupby('day_of_week')['total_amount'].sum().idxmax()
]

findings = [
    ('📦 Total Cleaned Transactions', f'{len(df):,}'),
    ('👥 Unique Customers',            f'{df["customer_id"].nunique():,}'),
    ('💰 Total Revenue',               f'£{df["total_amount"].sum():,.2f}'),
    ('⚠️  Overall Churn Rate',          f'{churn_rate:.1%}'),
    ('🏆 Largest Segment',             top_seg),
    ('🌍 Top Revenue Country',         top_country_rev),
    ('📅 Best Revenue Day',            best_day),
    ('🕙 Peak Purchase Hour',          '10:00 – 14:00'),
    ('📈 Pareto Ratio (top 20%)',      f'Approx 80% revenue'),
    ('📉 Key Churn Predictor',         'Recency (highest importance)'),
]

for label, value in findings:
    print(f'  {label:<40}: {value}')

print()
print('💡 RECOMMENDATIONS:')
print('  1. 🎯 Focus retention on "At Risk" and "Need Attention" segments')
print('  2. 📧 Send win-back campaigns to "Hibernating" customers < 180 days')
print('  3. 🏆 Reward "Champions" with loyalty perks to prevent churn')
print('  4. 📅 Schedule major promotions on Tue–Thu, 10:00–14:00')
print('  5. 🌍 Expand marketing in high-revenue non-UK markets')
print('  6. 📦 Bundle top-20 products for cross-sell upsell campaigns')
print('=' * 65)